In [ ]:
using Plots
using Serialization

In [ ]:
dir = "../../examples/ssa/runs/"

res0 = dir*"/solution_0res.jls"
x0, u_list0, H_used0, it0 = deserialize(res0)

res1 = dir*"/solution_1res.jls"
x1, u_list1, H_used1, it1 = deserialize(res1)

res2 = dir*"/solution_2res.jls"
x2, u_list2, H_used2, it2 = deserialize(res2)

# res3 = dir*"/solution_3res.jls"
# x3, u_list3, H_used3, it3 = deserialize(res3)

# res4 = dir*"/solution_4res.jls"
# x4, u_list4, H_used4, it4 = deserialize(res4)

In [ ]:
Base.@kwdef struct constants
    rho::Float64 = 900.0 # kg m^-3
    rhow::Float64 = 1000.0 # kg m^-3
    n::Float64 = 3.0 # for Glen's law
    B::Float64 = 1.9e8 # Pa s^1/3
    g::Float64 = 9.8 # m s^-2
    secpera::Float64 = 365*24*3600 # seconds per one year
end

# analytical solution
function analytical(x, cst::constants, ug, Hg, M0)
    rho = cst.rho # kg m^-3
    rhow = cst.rhow # kg m^-3
    n = cst.n # for Glen's law
    B = cst.B # Pa s^1/3
    g = cst.g # m s^-2
    secpera = cst.secpera
    ug = ug/secpera # m/s; speed at grouding line
    M0 = M0/secpera # m/s; surface mass balance

    # for convenience
    r = rho/rhow
    Cs = (0.25*rho*g*(1-r)/B)^n
    qg = ug*Hg

    # initiate u, H
    Nx = length(x)
    u = zeros(Nx)
    H = zeros(Nx)

    # form u, H
    for i in 1:Nx
        u[i] = ( ug^(n+1) + (Cs/M0) * ((M0 * x[i] + qg)^(n+1) - qg^(n+1)) )^(1/(n+1))
        H[i] = (M0 * x[i] + qg) / u[i];
    end

    return H, u
end

In [ ]:
# for analytical solution (and boundary condition at x=0)
ug = 50 # m/y; speed at grounding line
Hg = 500 # m; ice thickness at grounding line
M0 = 0.3 # m/y; surface mass balance
cst = constants()

# analytical sollution
H0, u0 = analytical(x0, cst, ug, Hg, M0)
H1, u1 = analytical(x1, cst, ug, Hg, M0)
H2, u2 = analytical(x2, cst, ug, Hg, M0)
# H3, u3 = analytical(x3, cst, ug, Hg, M0)
# H4, u4 = analytical(x4, cst, ug, Hg, M0)

In [ ]:
println(maximum(H0.-H_used0))
println(maximum(H1.-H_used1))
println(maximum(H2.-H_used2))
# println(maximum(H3.-H_used3))
# println(maximum(H4.-H_used4))

In [ ]:
plot(H2)

In [ ]:
err0 = u0.-u_list0[end];
println(maximum(err0)*cst.secpera)

plot(x0,u0)
plot!(x0, u_list0[1])
plot!(x0, u_list0[10])
plot!(x0, u_list0[end])

In [ ]:
err2 = u2.-u_list2[end];
println(maximum(err2)*cst.secpera)

plot(x2,u2)
#plot!(x2, u_list2[1])
plot!(x2, u_list2[10])
plot!(x2, u_list2[end])

In [ ]:
dx2 = x2[2]-x2[1]

In [ ]:
u2_x = (u2[end]-u2[end-1])/(dx2)
u_list2_x = (u_list2[end][end]-u_list2[end][end-1])/(dx2)

In [ ]:
u2_x-u_list2_x